In [51]:
import pandas as pd

In [52]:
df = pd.read_excel('../rico_data/from_framingham/All students assigned.xlsx')
df_schools = pd.read_csv('../rico_data/schools.csv')
school_names = df_schools['name'].tolist()
df = df[
    ['Student_Program', 
     'Student_School', 
     'BUS', 
     'P/U D/O TIME', 
     'BUS STOP']
    ].rename(columns={
        'Student_Program': 'program',
        'Student_School': 'school',
        'P/U D/O TIME': 'time',
        'BUS STOP': 'stop',
        'BUS': 'bus',
    })
df['has_sped'] = df['program'].str.contains('SPED', case=False, na=False)
df = df.drop(columns=['program'])
df = df[df['time'] <= pd.Timestamp('10:00:00').time()]
df = df[df['school'].isin(school_names)]


In [57]:
df['has_sped'] = df.groupby(['school', 'bus', 'time', 'stop'])['has_sped'].transform('any')
df = df.drop_duplicates(subset=['school', 'bus', 'time', 'stop'])
df = df.sort_values(by=['school', 'bus', 'time', 'stop']).reset_index(drop=True)
df.to_csv('../rico_data/current_schedule.csv', index=False)

In [56]:
# Create schedule overview
schedule_overview = df.groupby('bus').agg({'time': ['min', 'max'], 'stop': 'first', 'school': 'first', 'has_sped': 'any'})
schedule_overview.columns = [f"{col[0]}_{col[1]}" if col[1] not in ('first', 'any') else col[0] for col in schedule_overview.columns]
schedule_overview = schedule_overview.reset_index()

# Compute running time in minutes
time_min_td = pd.to_timedelta(schedule_overview['time_min'].astype(str))
time_max_td = pd.to_timedelta(schedule_overview['time_max'].astype(str))
schedule_overview['running_time_min'] = (time_max_td - time_min_td).dt.total_seconds() / 60

# Reorder and rename columns
schedule_overview = schedule_overview.reset_index()[[
    'bus', 
    'time_min', 
    'time_max',
    'running_time_min', 
    'stop',
    'school',
    'has_sped'
]].rename(columns={'time_min': 'start_time', 'time_max': 'end_time', 'stop': 'first_stop'})

# Sort by school and start time and save to CSV
schedule_overview = schedule_overview.sort_values(by=['school', 'start_time'])
schedule_overview.to_csv('../rico_data/schedule_overview.csv', index=False)